# População residente — total e variação anual

Motor de coleta e publicação das séries `populacao-total-anual` e `populacao-variacao-anual`.

- **Fonte:** IBGE — *Projeções da População, Revisão 2024* (população conciliada, data de
  referência em 1º de julho), tabela 4 "Indicadores Implícitos": `POP_T` (população total) e
  `TCG_T` (taxa de crescimento geométrico anual).
- **Endpoint:** `https://ftp.ibge.gov.br/Projecao_da_Populacao/Projecao_da_Populacao_2024/projecoes_2024_tab4_indicadores.xlsx`
- **Conferência:** `https://apisidra.ibge.gov.br/values/t/6579/n1/all/v/9324/p/all` — a tabela
  SIDRA 6579 ("População residente estimada") publica as Estimativas de População do DOU; onde
  ela e a projeção se sobrepõem, os valores têm de ser idênticos.
- **Definição:** população residente total do Brasil, em pessoas, estimada para 1º de julho de
  cada ano; e a variação percentual dessa população em relação ao ano anterior.
- **Fórmula:** nenhuma para o total — o valor é lido de `POP_T`. A variação é a taxa de
  crescimento geométrico que o IBGE publica no mesmo arquivo (`TCG_T`), idêntica a
  `(P_t / P_(t-1) − 1) × 100`; a identidade é conferida ponto a ponto na última célula.
- **Período publicado:** `2000-01-01` a `2026-01-01` (a variação começa em `2001-01-01`). A
  revisão 2024 não tem série conciliada antes de 2000, então 1995–1999 — que a série antiga
  publicava por interpolação — saem.
- **Arquivos gerados:** `indicadores/populacao/populacao-total-anual.json` e
  `indicadores/populacao/populacao-variacao-anual.json`.

## Qual população esta série publica

Havia três candidatas, e todas se chamam "população":

1. **A série publicada até aqui** (`ipeadatapy.timeseries('DEPIS_POP')`) não é do IBGE: é a série
   *População* do IPEADATA, construída por **interpolação spline cúbica** entre os Censos e
   Contagens de 1872 a 2022 — é o que diz a ficha da própria série. Só os anos de Censo ou de
   Contagem são medidos; todo o resto é curva desenhada entre eles. Foi essa curva que produziu a queda de
   2020–2022 (204,08 → 203,79 → 203,08 milhões, −0,3% em dois anos) num país que não encolheu, e
   a retomada de 2023–2024, que é extrapolação.
2. **As Estimativas de População do IBGE** (SIDRA 6579) são a série oficial publicada no DOU todo
   agosto, com data de referência em 1º de julho. Mas a tabela é um mosaico de safras: 2001–2021
   são as estimativas *como publicadas na época*, com o degrau de 2013 (193,9 → 201,0 milhões em
   um ano, troca de base metodológica), e faltam 2007, 2010, 2022 e 2023; 2024–2026 já são a
   revisão de 2024. Usá-la como série contínua é publicar degraus artificiais.
3. **As Projeções da População — Revisão 2024** (aqui adotadas) são a base vigente que o IBGE
   recomenda para populações anuais e a origem das estimativas publicadas desde 2024: uma única
   metodologia (componentes demográficas) para 2000–2070, conciliada com os Censos de 2000, 2010
   e 2022, os registros vitais e a Pesquisa de Pós-Enumeração de 2022.

## O Censo 2022 e o degrau que ele não criou

A revisão 2024 não costura a estimativa antiga com o Censo: reconstrói a série inteira sobre uma
população de partida **conciliada**. Em 1º de julho de 2022 a população recenseada era
202.952.784 e a conciliada é **210.862.983** (+3,9%) — o IBGE ajusta a cobertura do Censo
(Pesquisa de Pós-Enumeração) e a consistência com nascimentos, óbitos e migração, e aplica o
mesmo tratamento a 2000 (+3,0%) e 2010 (+2,2%). O número do Censo **não** é revisado: 203.080.756
(1º de agosto de 2022) continua sendo o publicado; são duas medidas diferentes, e a série anual
usa a conciliada. O efeito prático é não haver degrau em 2022: o crescimento de 2021 para 2022 é
+0,36%, não a queda que a interpolação do IPEA produzia.

## Data de referência

O valor é de **1º de julho** do ano; a data publicada é `AAAA-01-01`. O período do contrato (C9) é
o ano, e o dia 01 é a convenção de todas as séries anuais do repositório (`pib-real-anual`,
`homicidios-anual`) — é o que mantém as séries alinhadas no mesmo eixo de tempo do app. Carimbar
`AAAA-07-01` seria mais literal e deslocaria os 27 pontos meio ano em relação às outras séries
anuais (a matriz de cobertura do app não muda: ela conta pontos por ano), mas é mudança de
convenção do contrato, não desta série. Fica registrado como decisão pendente do dono do repo;
por ora a convenção é documentada, não alterada.

## A variação anual

O IBGE publica a variação pronta no mesmo arquivo: `TCG_T`, a taxa de crescimento geométrico
anual. Ela coincide com `(P_t / P_(t-1) − 1) × 100` — a célula final prova a identidade em todos
os 26 pontos (maior resíduo na ordem de 1e-15, erro de ponto flutuante). Publicar a taxa da fonte,
calculada sobre a mesma série do total, é melhor do que derivar por fora: assim não há como as
duas séries divergirem. O primeiro ponto é 2001, porque 2000 é o primeiro ano da revisão e serve
de denominador.

## Execução sem sobrescrever o publicado

A variável de ambiente `POPULACAO_SAIDA` redireciona a pasta de saída (padrão:
`indicadores/populacao`). Para rodar em ensaio:

```
POPULACAO_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook População.ipynb
```

## Coleta

A planilha de projeções é baixada inteira a cada execução. Não há leitura do JSON já publicado
nem atualização incremental: a série é recalculada do zero.

In [1]:
from datetime import date, datetime
from pathlib import Path
import io
import json
import math
import os
import time
import xml.etree.ElementTree as ET
import zipfile

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
PASTA = RAIZ / "indicadores" / "populacao" if (RAIZ / "indicadores" / "populacao").is_dir() else RAIZ
DESTINO = Path(os.environ.get("POPULACAO_SAIDA", PASTA))

URL_PROJECAO = (
    "https://ftp.ibge.gov.br/Projecao_da_Populacao/Projecao_da_Populacao_2024/"
    "projecoes_2024_tab4_indicadores.xlsx"
)
URL_SIDRA = "https://apisidra.ibge.gov.br/values/t/6579/n1/all/v/9324/p/all"
URL_SIDRA_PERIODOS = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos"
REVISAO = date(2024, 8, 22)  # divulgação das Projeções da População, Revisão 2024

PRIMEIRO_ANO = 2000  # primeiro ano da Revisão 2024 (antes disso não há série conciliada)
NS = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
LIGACAO = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"


def baixar(url, tentativas=4, formato="json", **parametros):
    """GET com repetição e espera crescente — os servidores do IBGE falham esporadicamente.

    `formato` diz o que a resposta precisa conter (`json` ou `xlsx`). Resposta 200 com HTML no
    lugar do conteúdo esperado também é repetida: acontece nos servidores do IBGE.
    """
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(url, timeout=180, **parametros)
            resposta.raise_for_status()
            if formato == "json":
                resposta.json()
            elif formato == "xlsx" and not resposta.content.startswith(b"PK"):
                raise ValueError(f"{url} nao devolveu um .xlsx")
            return resposta
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)


def ler_xlsx(conteudo):
    """Linhas da primeira planilha do .xlsx como dicionários coluna -> texto.

    O .xlsx é um zip de XML; ler com a biblioteca padrão evita depender de `openpyxl`, que não
    está nas dependências fixadas do repositório.
    """
    with zipfile.ZipFile(io.BytesIO(conteudo)) as pacote:
        compartilhadas = []
        if "xl/sharedStrings.xml" in pacote.namelist():
            for item in ET.fromstring(pacote.read("xl/sharedStrings.xml")):
                compartilhadas.append("".join(t.text or "" for t in item.iter(NS + "t")))
        livro = ET.fromstring(pacote.read("xl/workbook.xml"))
        folha = next(iter(livro.iter(NS + "sheet")))
        relacoes = ET.fromstring(pacote.read("xl/_rels/workbook.xml.rels"))
        alvo = next(r.get("Target") for r in relacoes if r.get("Id") == folha.get(LIGACAO))
        caminho = alvo.lstrip("/")
        if not caminho.startswith("xl/"):
            caminho = f"xl/{caminho}"
        raiz = ET.fromstring(pacote.read(caminho))

    linhas = []
    for linha in raiz.iter(NS + "row"):
        celulas = {}
        for celula in linha.iter(NS + "c"):
            valor = celula.find(NS + "v")
            if valor is None:
                continue
            coluna = "".join(ch for ch in celula.get("r") if ch.isalpha())
            celulas[coluna] = (
                compartilhadas[int(valor.text)] if celula.get("t") == "s" else valor.text
            )
        linhas.append(celulas)
    return linhas


linhas = ler_xlsx(baixar(URL_PROJECAO, formato="xlsx").content)
inicio_tabela = next(
    i for i, linha in enumerate(linhas) if linha.get("A") == "ANO" and linha.get("E") == "POP_T"
)

projecao = {}
for linha in linhas[inicio_tabela + 1 :]:
    if linha.get("C") == "BR" and str(linha.get("A", "")).isdigit():
        # 2000 é o primeiro ano da série e não tem taxa de crescimento (não há 1999).
        taxa = float(linha["H"]) if linha.get("H") else None
        projecao[int(linha["A"])] = (int(linha["E"]), taxa)

if not projecao:
    raise ValueError("nenhuma linha do Brasil encontrada na planilha de projeção")

anos = sorted(projecao)
if anos != list(range(anos[0], anos[-1] + 1)):
    raise ValueError(f"a projeção tem ano faltando: {anos[0]}..{anos[-1]}")
if projecao[2022][0] != 210_862_983:
    raise ValueError(f"2022 fora do esperado: {projecao[2022][0]}")

oficiais = {
    int(linha["D3C"]): int(linha["V"])
    for linha in baixar(URL_SIDRA).json()[1:]
    if str(linha.get("V", "")).isdigit()
}
# A tabela 6579 guarda safras diferentes: 2001-2021 são as estimativas como publicadas na
# época, e só os períodos republicados depois da Revisão 2024 pertencem à série atual.
republicados = {
    int(periodo["id"])
    for periodo in baixar(URL_SIDRA_PERIODOS).json()
    if datetime.strptime(periodo["modificacao"], "%d/%m/%Y").date() >= REVISAO
}
ULTIMO_ANO = max(oficiais)
if ULTIMO_ANO > date.today().year:
    raise ValueError(f"última estimativa publicada no futuro: {ULTIMO_ANO}")
faltando = [ano for ano in range(PRIMEIRO_ANO + 1, ULTIMO_ANO + 1) if projecao[ano][1] is None]
if faltando:
    raise ValueError(f"a projeção não traz TCG_T para: {faltando}")

print(f"Projeção da População (Revisão 2024): {anos[0]} a {anos[-1]}, {len(anos)} anos")
print(f"Estimativas publicadas (SIDRA 6579): {min(oficiais)} a {ULTIMO_ANO}, {len(oficiais)} anos")
print(f"Anos da SIDRA republicados na Revisão 2024: {sorted(republicados)[0]}..{max(republicados)}")
print(f"Anos publicados aqui: {PRIMEIRO_ANO} a {ULTIMO_ANO}")
print(f"Pasta de saída: {DESTINO}")

Projeção da População (Revisão 2024): 2000 a 2070, 71 anos
Estimativas publicadas (SIDRA 6579): 2001 a 2026, 22 anos
Anos da SIDRA republicados na Revisão 2024: 2024..2026
Anos publicados aqui: 2000 a 2026
Pasta de saída: /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/populacao


## Recorte, conferência e variação

A série publicada vai de 2000 — primeiro ano da revisão, quando a conciliação começa — até a
última estimativa publicada no DOU. Os anos que a SIDRA 6579 já publica são conferidos contra a
projeção: divergência ali significa que o IBGE lançou uma revisão nova e a série precisa ser
revisada de novo.

A variação é `TCG_T` do próprio arquivo. A conferência abaixo recalcula
`(P_t / P_(t-1) − 1) × 100` sobre `POP_T` e mede o resíduo — não para substituir o valor da fonte,
mas para provar que ela usa exatamente a mesma série do total.

In [2]:
publicados = [(ano, *projecao[ano]) for ano in range(PRIMEIRO_ANO, ULTIMO_ANO + 1)]

conferidos = sorted(
    ano for ano in republicados if ano in oficiais and PRIMEIRO_ANO <= ano <= ULTIMO_ANO
)
if ULTIMO_ANO not in conferidos:
    raise ValueError(f"{ULTIMO_ANO} não está entre as estimativas republicadas na Revisão 2024")
for ano in conferidos:
    if projecao[ano][0] != oficiais[ano]:
        raise ValueError(f"{ano}: projeção {projecao[ano][0]} != SIDRA 6579 {oficiais[ano]}")

total = [(ano, valor) for ano, valor, _ in publicados]
variacao = [(ano, taxa) for ano, _, taxa in publicados if ano > PRIMEIRO_ANO]
por_ano = dict(total)

residuos = [
    (ano, (valor / por_ano[ano - 1] - 1) * 100 - taxa)
    for ano, valor, taxa in publicados
    if ano > PRIMEIRO_ANO
]
maior_residuo = max(residuos, key=lambda item: abs(item[1]))

print(f"Anos conferidos contra a SIDRA 6579: {conferidos[0]}..{conferidos[-1]} ({len(conferidos)})")
print(f"Pontos do total: {len(total)} ({total[0][0]}..{total[-1][0]})")
print(f"Pontos da variação: {len(variacao)} ({variacao[0][0]}..{variacao[-1][0]})")
print(f"Maior resíduo |TCG_T - derivada|: {maior_residuo[1]:.2e} p.p. em {maior_residuo[0]}")
print()
for ano, valor, taxa in publicados:
    if ano in (2000, 2001, 2010, 2020, 2022, 2023, 2024, 2025, 2026):
        rotulo = "—" if taxa is None else f"{taxa:+.4f}%"
        print(f"  {ano}  {valor:>12,} hab.  {rotulo}")
print()
print(f"População 2022 conciliada: {por_ano[2022]:,} hab. "
      f"({por_ano[2022] / 203_080_756 - 1:+.2%} sobre o Censo de 1º/8/2022, 203.080.756)")
print(f"Menor variação anual: {min(variacao, key=lambda item: item[1])}")
print(f"Maior variação anual: {max(variacao, key=lambda item: item[1])}")

Anos conferidos contra a SIDRA 6579: 2024..2026 (3)
Pontos do total: 27 (2000..2026)
Pontos da variação: 26 (2001..2026)
Maior resíduo |TCG_T - derivada|: -4.22e-15 p.p. em 2006

  2000   174,695,935 hab.  —
  2001   177,003,743 hab.  +1.3210%
  2010   194,749,329 hab.  +0.8155%
  2020   209,164,889 hab.  +0.6084%
  2022   210,862,983 hab.  +0.3614%
  2023   211,695,158 hab.  +0.3947%
  2024   212,583,750 hab.  +0.4198%
  2025   213,421,037 hab.  +0.3939%
  2026   214,211,951 hab.  +0.3706%

População 2022 conciliada: 210,862,983 hab. (+3.83% sobre o Censo de 1º/8/2022, 203.080.756)
Menor variação anual: (2022, 0.361412583224041)
Maior variação anual: (2001, 1.32104275923763)


## Escrita dos JSONs do contrato

Os dois arquivos são gravados inteiros (nada de merge com o que está publicado), com `versao`,
`id`, `unidade`, `eixo`, `medida`, `granularidade`, `fonte`, `urlFonte` e `dados` no formato
`{"data": "YYYY-MM-DD", "valor": ...}`.

O total é inteiro (habitantes). A variação vai **sem arredondamento**: é o `TCG_T` do IBGE, e
arredondar quebraria a identidade com o total; a exibição usa 2 casas (`unidade.casas`).

In [3]:
serie_total = {
    "versao": 1,
    "id": "populacao-total-anual",
    "unidade": {"tipo": "numero", "simbolo": "hab.", "casas": 0},
    "eixo": "contagem",
    "medida": "estoque",
    "granularidade": "anual",
    "fonte": "IBGE",
    "urlFonte": URL_PROJECAO,
    "dados": [{"data": f"{ano}-01-01", "valor": valor} for ano, valor in total],
}

serie_variacao = {
    "versao": 1,
    "id": "populacao-variacao-anual",
    "unidade": {"tipo": "percentual", "simbolo": "%", "casas": 2},
    "eixo": "percentual",
    "medida": "variacao-anual",
    "granularidade": "anual",
    "fonte": "IBGE",
    "urlFonte": URL_PROJECAO,
    "dados": [{"data": f"{ano}-01-01", "valor": taxa} for ano, taxa in variacao],
}

DESTINO.mkdir(parents=True, exist_ok=True)
for serie in (serie_total, serie_variacao):
    caminho = DESTINO / f"{serie['id']}.json"
    with caminho.open("w", encoding="utf-8") as arquivo:
        json.dump(serie, arquivo, ensure_ascii=False, indent=2)
        arquivo.write("\n")
    print(f"{len(serie['dados'])} pontos escritos em {caminho}")

27 pontos escritos em /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/populacao/populacao-total-anual.json
26 pontos escritos em /Users/lucas/Workspace/personal/brasilemnumeros/dados/indicadores/populacao/populacao-variacao-anual.json


## Validação inline

A célula falha se o contrato, a ordem, a unicidade e a continuidade dos anos, a finitude dos
valores, a identidade entre variação e total, ou a igualdade com a fonte forem violados. Ela
também confere os números citados em `docs/metodologia.md`.

In [4]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}

for serie, identificador, unidade, eixo, medida in (
    (serie_total, "populacao-total-anual", {"tipo": "numero", "simbolo": "hab.", "casas": 0}, "contagem", "estoque"),
    (serie_variacao, "populacao-variacao-anual", {"tipo": "percentual", "simbolo": "%", "casas": 2}, "percentual", "variacao-anual"),
):
    assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
    assert serie["versao"] == 1, serie["versao"]
    assert serie["id"] == identificador, serie["id"]
    assert serie["unidade"] == unidade, serie["unidade"]
    assert serie["eixo"] == eixo, serie["eixo"]
    assert serie["medida"] == medida, serie["medida"]
    assert serie["granularidade"] == "anual", serie["granularidade"]
    assert serie["fonte"] == "IBGE", serie["fonte"]
    assert serie["urlFonte"].startswith("https://"), serie["urlFonte"]
    assert serie["dados"], "série vazia"

    datas = [ponto["data"] for ponto in serie["dados"]]
    dias = [date.fromisoformat(texto) for texto in datas]
    for ponto in serie["dados"]:
        assert set(ponto) == {"data", "valor"}, sorted(ponto)
        assert isinstance(ponto["data"], str), ponto
        assert not isinstance(ponto["valor"], bool), ponto
        assert isinstance(ponto["valor"], (int, float)), ponto
        assert math.isfinite(ponto["valor"]), ponto

    assert all(a < b for a, b in zip(dias, dias[1:])), f"{identificador}: datas fora de ordem"
    assert all(dia.day == 1 and dia.month == 1 for dia in dias), f"{identificador}: data diferente de 01-01"
    assert all(b.year - a.year == 1 for a, b in zip(dias, dias[1:])), f"{identificador}: ano faltando"

    with (DESTINO / f"{identificador}.json").open(encoding="utf-8") as arquivo:
        assert json.load(arquivo) == serie, f"{identificador}: arquivo gravado difere do objeto validado"

assert len(serie_total["dados"]) == ULTIMO_ANO - PRIMEIRO_ANO + 1
assert len(serie_variacao["dados"]) == len(serie_total["dados"]) - 1
assert serie_total["dados"][0]["data"] == "2000-01-01"
assert serie_total["dados"][-1]["data"] == f"{ULTIMO_ANO}-01-01"
assert serie_variacao["dados"][0]["data"] == "2001-01-01"

# 1) Cada ponto é o da fonte, sem interpolação e sem merge.
assert [(p["data"], p["valor"]) for p in serie_total["dados"]] == [
    (f"{ano}-01-01", valor) for ano, valor in total
]
assert [(p["data"], p["valor"]) for p in serie_variacao["dados"]] == [
    (f"{ano}-01-01", taxa) for ano, taxa in variacao
]

# 2) A variação é (P_t/P_(t-1) - 1) * 100 em TODOS os pontos (resíduo só de ponto flutuante).
totais = {p["data"][:4]: p["valor"] for p in serie_total["dados"]}
for ponto in serie_variacao["dados"]:
    ano = int(ponto["data"][:4])
    derivada = (totais[str(ano)] / totais[str(ano - 1)] - 1) * 100
    assert abs(ponto["valor"] - derivada) < 1e-9, (ano, ponto["valor"], derivada)

# 3) A variação usa a MESMA série do total: o valor da fonte é o TCG_T de POP_T.
for ano, valor, taxa in publicados:
    if ano > PRIMEIRO_ANO:
        assert abs(taxa - (valor / por_ano[ano - 1] - 1) * 100) < 1e-9, ano

# 4) Não há degrau: a população cresce em todos os anos (a queda de 2020-2022 era artefato).
assert all(taxa > 0 for _, taxa in variacao), "variação negativa — revisar a fonte"

# 5) Números citados em docs/metodologia.md.
assert totais["2000"] == 174_695_935
assert totais["2010"] == 194_749_329
assert totais["2020"] == 209_164_889
assert totais["2022"] == 210_862_983
assert totais["2023"] == 211_695_158
assert totais["2024"] == 212_583_750
variacao_por_ano = {p["data"][:4]: p["valor"] for p in serie_variacao["dados"]}
assert abs(variacao_por_ano["2022"] - 0.3614) < 1e-4
assert abs(variacao_por_ano["2001"] - 1.3210) < 1e-4

print(
    f"ok: {len(total)} anos de população ({total[0][0]}..{total[-1][0]}) e "
    f"{len(variacao)} de variação, fiéis à Revisão 2024 do IBGE; "
    f"último valor {total[-1][1]:,} hab."
)

ok: 27 anos de população (2000..2026) e 26 de variação, fiéis à Revisão 2024 do IBGE; último valor 214,211,951 hab.
